# Notebook 04 — Bootstrap, Monte Carlo y potencia estadística | AWS SageMaker

**Proyecto:** A/B Testing de tasa de conversión (`control` vs. `treatment`).  
**Entrada:** `s3://TU_BUCKET/processed/ab_data_clean.csv` producido por el Notebook 01.  
**Salidas:** `results/notebook04/` con tablas CSV, gráficos PNG y un resumen JSON.

## Objetivos
1. Estimar la incertidumbre de la diferencia de conversiones con **bootstrap**.
2. Simular la distribución nula del estadístico mediante **Monte Carlo**.
3. Estimar potencia, error tipo I, error tipo II y tamaño de muestra antes de un experimento futuro.
4. Separar significancia estadística, incertidumbre y relevancia práctica.

> **Importante:** el dataset limpio excluye ciertos registros tras observar su página asignada. Es un análisis simplificado tipo *per-protocol*; la aleatorización original y un posible análisis *intention-to-treat* deben verificarse para inferencias causales robustas. Los resultados de simulación dependen de sus supuestos, no son observaciones adicionales.

## 0. Configuración y dependencias
Ejecuta las celdas secuencialmente. Configura `BUCKET`; si trabajas sin AWS utiliza `USE_S3 = False` y el archivo local.

In [ ]:
# Descomenta si fuese necesario en SageMaker:
# %pip install numpy pandas scipy statsmodels matplotlib boto3
from pathlib import Path
from io import BytesIO
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from statsmodels.stats.proportion import (proportions_ztest, confint_proportions_2indep, proportion_effectsize)
from statsmodels.stats.power import NormalIndPower
from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)
ALPHA = 0.05
N_BOOT = 10_000
N_MONTE_CARLO = 10_000
BUCKET = 'TU_BUCKET'  # <-- Sustituir
USE_S3 = True
INPUT_KEY = 'processed/ab_data_clean.csv'
LOCAL_PATH = Path('data/ab_data_clean.csv')
OUTPUT_PREFIX = 'results/notebook04/'
OUTPUT_DIR = Path('outputs/notebook04')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Carga de datos y control de calidad
Confirmamos una sola observación por usuario y conversiones binarias. El grupo `control` siempre será A y `treatment` siempre será B.

In [ ]:
if USE_S3:
    if BUCKET == 'TU_BUCKET':
        raise ValueError('Configura BUCKET con el bucket de S3 antes de continuar.')
    import boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=BUCKET, Key=INPUT_KEY)
    df = pd.read_csv(BytesIO(obj['Body'].read()))
else:
    df = pd.read_csv(LOCAL_PATH)

required = {'user_id','group','converted','landing_page'}
assert required.issubset(df.columns), f'Faltan columnas: {required - set(df.columns)}'
assert df['user_id'].is_unique, 'Revisar duplicados en Notebook 01'
assert not df[list(required)].isna().any().any(), 'Se detectaron nulos'
assert df['group'].isin(['control','treatment']).all(), 'Grupos inesperados'
assert df['converted'].isin([0,1]).all(), 'La conversión debe ser binaria'
assert (((df.group=='control') & (df.landing_page=='old_page')) | ((df.group=='treatment') & (df.landing_page=='new_page'))).all()

summary = (df.groupby('group')['converted'].agg(usuarios='size', conversiones='sum', tasa='mean')
           .reindex(['control','treatment']))
assert summary['usuarios'].gt(0).all()
n_a, n_b = summary['usuarios'].to_numpy(dtype=int)
x_a, x_b = summary['conversiones'].to_numpy(dtype=int)
p_a, p_b = summary['tasa'].to_numpy(dtype=float)
observed_diff = p_b-p_a
display(summary)
print(f'Diferencia observada B-A: {observed_diff*100:.4f} puntos porcentuales')

## 2. Bootstrap de la diferencia de proporciones

El **bootstrap no paramétrico** extrae repetidamente usuarios **con reemplazo y por separado dentro de cada grupo**, manteniendo fijo el tamaño de cada grupo. Calcula una nueva diferencia de tasas en cada réplica.

Como los datos son Bernoulli, remuestrear las etiquetas binarias es matemáticamente equivalente a generar `Binomial(n, p̂)/n` para cada grupo: así evitamos matrices enormes que consumirían memoria en SageMaker. Esta equivalencia depende de que las observaciones se traten como independientes.

**Pregunta de práctica:** ¿por qué remuestrear ambos grupos juntos cambiaría el experimento que queremos representar?

In [ ]:
boot_a = rng.binomial(n=n_a, p=p_a, size=N_BOOT)/n_a
boot_b = rng.binomial(n=n_b, p=p_b, size=N_BOOT)/n_b
boot_diff = boot_b - boot_a
ci_boot = np.quantile(boot_diff, [ALPHA/2, 1-ALPHA/2])
ci_newcombe = confint_proportions_2indep(x_b,n_b,x_a,n_a,
                                          compare='diff', method='newcomb',alpha=ALPHA)
print(f'Efecto observado: {observed_diff*100:.4f} pp')
print(f'IC bootstrap percentil 95%: [{ci_boot[0]*100:.4f}, {ci_boot[1]*100:.4f}] pp')
print(f'IC Newcombe 95%: [{ci_newcombe[0]*100:.4f}, {ci_newcombe[1]*100:.4f}] pp')
print('¿Bootstrap incluye cero?:',ci_boot[0]<=0<=ci_boot[1])

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.hist(100*boot_diff,bins=55,alpha=.80,edgecolor='white')
ax.axvline(100*observed_diff,color='black',lw=2,label='Diferencia observada')
ax.axvline(0,color='red',ls='--',label='Diferencia nula')
for v in ci_boot:
    ax.axvline(100*v,color='darkorange',ls=':')
ax.set(xlabel='Diferencia B − A (puntos porcentuales)',ylabel='Frecuencia',title='Bootstrap: distribución de la diferencia de conversión')
ax.legend();plt.tight_layout()
fig.savefig(OUTPUT_DIR/'bootstrap_difference.png',dpi=150)
plt.show()

### Interpretación
- Si el intervalo incluye **0**, los datos son compatibles con ausencia de diferencia; esto **no prueba equivalencia**.
- Si queda por encima o por debajo de 0, sugiere un efecto con signo distinto de cero con este procedimiento.
- El IC percentil bootstrap puede diferir del IC Newcombe; diferencias pequeñas son esperables.

**Ejercicio:** modifica `N_BOOT` a 2,000 y 20,000; describe cómo cambia la estabilidad de los extremos del intervalo.

## 3. Monte Carlo bajo la hipótesis nula

**H₀:** las tasas poblacionales son iguales. Estimamos una probabilidad común `p_pool` y generamos experimentos sintéticos de igual tamaño que el observado. Calculamos el estadístico Z en cada réplica con el error estándar combinado estimado en esa réplica.

El p-value de Monte Carlo será la fracción de réplicas con `|Z_sim| >= |Z_obs|`, con corrección de +1 para evitar reportar cero. **Es una simulación paramétrica de la hipótesis nula, no un test exacto de aleatorización.**

In [ ]:
z_obs, p_z = proportions_ztest([x_b,x_a],[n_b,n_a],alternative='two-sided')
p_pool = (x_a+x_b)/(n_a+n_b)
null_a = rng.binomial(n_a,p_pool,size=N_MONTE_CARLO)
null_b = rng.binomial(n_b,p_pool,size=N_MONTE_CARLO)
null_pa = null_a/n_a
null_pb = null_b/n_b
null_pool=(null_a+null_b)/(n_a+n_b)
null_se=np.sqrt(null_pool*(1-null_pool)*(1/n_a+1/n_b))
null_z=np.divide(null_pb-null_pa,null_se,out=np.zeros_like(null_pa),where=null_se>0)
p_mc=(1+np.count_nonzero(np.abs(null_z)>=abs(z_obs)))/(N_MONTE_CARLO+1)
se_p_mc=np.sqrt(p_mc*(1-p_mc)/(N_MONTE_CARLO+1))
print(f'Z observado: {z_obs:.4f}')
print(f'p-value analítico bilateral: {p_z:.6f}')
print(f'p-value Monte Carlo bilateral: {p_mc:.6f} (EE MC ≈ {se_p_mc:.6f})')
print('Decisión (Z):', 'rechazar H0' if p_z<ALPHA else 'no rechazar H0')

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
ax.hist(null_z,bins=55,density=True,alpha=.7,label='Z simulado bajo H0')
xx=np.linspace(-4,4,400)
ax.plot(xx,norm.pdf(xx),label='Normal estándar aproximada')
ax.axvline(z_obs,color='crimson',ls='--',label='Z observado')
ax.axvline(-abs(z_obs),color='crimson',ls=':')
ax.set(xlabel='Estadístico Z',ylabel='Densidad',title='Distribución nula y estadístico observado')
ax.legend();plt.tight_layout()
fig.savefig(OUTPUT_DIR/'monte_carlo_null.png',dpi=150);plt.show()

### Interpretación de p-values
El p-value **no** es la probabilidad de que H₀ sea verdadera. Expresa cuán extremo sería el resultado si H₀ fuese cierta, dadas las suposiciones del procedimiento.

**Ejercicio:** ¿por qué puede variar un poco `p_mc` al modificar la semilla o el número de simulaciones?

## 4. Errores tipo I, tipo II y potencia

- **Tipo I (α):** rechazamos H₀ cuando en realidad no existe efecto.
- **Tipo II (β):** no rechazamos H₀ cuando sí existe un efecto concreto.
- **Potencia (1−β):** probabilidad de detectar ese efecto.

Para planear un **experimento futuro** fijamos antes del análisis una conversión base de 12%, un objetivo del 13% (+1 pp), α=0.05, prueba bilateral y potencia del 80%. Estos valores son **hipotéticos**, no se infieren del dataset.

In [ ]:
BASELINE = 0.12
MDE_PP = 1.0
TREATMENT_EXPECTED = BASELINE + MDE_PP/100
TARGET_POWER = 0.80
assert 0<BASELINE<1 and 0<TREATMENT_EXPECTED<1
h = abs(proportion_effectsize(TREATMENT_EXPECTED,BASELINE))
power_model=NormalIndPower()
n_per_group=int(np.ceil(power_model.solve_power(effect_size=h, alpha=ALPHA,
                                     power=TARGET_POWER, ratio=1, alternative='two-sided')))
print('Usuarios necesarios por grupo (aproximación normal):',n_per_group)
print('Total (asignación 1:1):',2*n_per_group)

## 5. Simulación de poder estadístico

Simulamos experimentos independientes y registramos cuántos rechazan H₀ con la misma prueba Z bilateral. Medimos:

- Falsos positivos cuando `p_A = p_B` → tasa empírica tipo I.
- Detecciones cuando `p_B = p_A + MDE` → potencia empírica.
- No detecciones bajo efecto real → error tipo II empírico.

Para evitar lentitud usamos operaciones vectorizadas.

In [ ]:
def simulated_rejection_rate(n_a, n_b, p_a, p_b, trials=5000, alpha=.05, seed=42):
    if min(n_a,n_b)<=0 or not (0<p_a<1 and 0<p_b<1):
        raise ValueError('Tamaños positivos y probabilidades entre 0 y 1')
    generator=np.random.default_rng(seed)
    xa=generator.binomial(n_a,p_a,size=trials)
    xb=generator.binomial(n_b,p_b,size=trials)
    pooled=(xa+xb)/(n_a+n_b)
    se=np.sqrt(pooled*(1-pooled)*(1/n_a+1/n_b))
    z=np.divide(xb/n_b-xa/n_a,se,out=np.zeros(trials,dtype=float),where=se>0)
    pvals=2*norm.sf(np.abs(z))
    return float(np.mean(pvals<alpha))

emp_alpha=simulated_rejection_rate(n_per_group,n_per_group,BASELINE,BASELINE,seed=10)
emp_power=simulated_rejection_rate(n_per_group,n_per_group,BASELINE,TREATMENT_EXPECTED,seed=11)
print(f'Error tipo I simulado: {emp_alpha:.3%} (objetivo {ALPHA:.0%})')
print(f'Potencia simulada: {emp_power:.3%} (objetivo {TARGET_POWER:.0%})')
print(f'Error tipo II simulado: {1-emp_power:.3%}')

## 6. Curva de potencia vs. tamaño de muestra
**Ejercicio:** identifica el tamaño de muestra más cercano al 80% y explica por qué duplicar usuarios no duplica la potencia.

In [ ]:
sample_sizes=np.unique(np.rint(np.linspace(1000,max(2500,n_per_group*1.8),14)).astype(int))
power_curve=[simulated_rejection_rate(int(n),int(n),BASELINE,TREATMENT_EXPECTED,
                                     trials=3000,seed=100+i) for i,n in enumerate(sample_sizes)]
curve_df=pd.DataFrame({'usuarios_por_grupo':sample_sizes,'potencia_simulada':power_curve})
display(curve_df)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(sample_sizes,np.array(power_curve)*100,marker='o')
ax.axhline(80,color='gray',ls='--',label='Objetivo 80%')
ax.axvline(n_per_group,color='darkorange',ls=':',label='n aproximado para 80%')
ax.set(xlabel='Usuarios por grupo',ylabel='Potencia estimada (%)',title='Potencia estadística simulada',ylim=(0,105))
ax.legend();plt.tight_layout()
fig.savefig(OUTPUT_DIR/'power_curve.png',dpi=150);plt.show()

## 7. Sensibilidad a la mejora mínima detectable (MDE)

Si queremos detectar diferencias más pequeñas, necesitamos muestras más grandes. La MDE debe elegirse según un beneficio de negocio relevante, no para ajustar retrospectivamente el resultado observado.

In [ ]:
mde_values=[0.25,0.5,1.0,1.5,2.0]
records=[]
for pp in mde_values:
    candidate=BASELINE+pp/100
    es=abs(proportion_effectsize(candidate,BASELINE))
    nn=int(np.ceil(power_model.solve_power(effect_size=es,alpha=ALPHA,power=TARGET_POWER,alternative='two-sided')))
    records.append({'MDE_pp':pp,'conversion_esperada':candidate,'usuarios_por_grupo':nn,'usuarios_totales':2*nn})
mde_df=pd.DataFrame(records)
display(mde_df)

## 8. Verificación de supuestos y decisiones de negocio

Antes de recomendar el despliegue, revisar:

1. ¿Hubo verdadera aleatorización y asignación estable a nivel usuario?
2. ¿Existe interferencia entre usuarios o conversiones repetidas?
3. ¿La limpieza alteró diferencialmente a grupos A y B?
4. ¿Se definió de antemano la métrica principal y duración del experimento? Evitar detener el experimento cuando el p-value pasa de 0.05.
5. ¿El intervalo de diferencia descarta pérdidas de negocio importantes? Un resultado no significativo no implica equivalencia.

**Ejercicio escrito:** describe por qué un p-value no significativo puede coexistir con potencia insuficiente.

## 9. Guardar resultados en local y S3
No subimos datos sensibles. Las salidas contienen agregados y figuras. Necesitarás permisos `s3:PutObject` en el prefijo de resultados.

In [ ]:
results = {
    'seed': SEED,'alpha': ALPHA,'bootstrap_replicas': N_BOOT,'monte_carlo_replicas': N_MONTE_CARLO,
    'n_control':int(n_a),'n_treatment':int(n_b),
    'conversion_control':float(p_a),'conversion_treatment':float(p_b),
    'diff_B_minus_A':float(observed_diff),
    'bootstrap_ci_lower':float(ci_boot[0]),'bootstrap_ci_upper':float(ci_boot[1]),
    'newcombe_ci_lower':float(ci_newcombe[0]),'newcombe_ci_upper':float(ci_newcombe[1]),
    'z_statistic':float(z_obs),'p_value_z':float(p_z),'p_value_mc':float(p_mc),
    'planning_baseline':BASELINE,'planning_mde_pp':MDE_PP,
    'planning_target_power':TARGET_POWER,'planning_n_per_group':n_per_group,
    'simulated_type_I_rate':emp_alpha,'simulated_power':emp_power,
    'interpretation_note':'Muestra limpiada per-protocol; causalidad requiere validar asignación y exclusiones.'
}
with open(OUTPUT_DIR/'statistical_summary.json','w',encoding='utf-8') as f:
    json.dump(results,f,ensure_ascii=False,indent=2)
curve_df.to_csv(OUTPUT_DIR/'power_curve.csv',index=False)
mde_df.to_csv(OUTPUT_DIR/'sample_size_by_mde.csv',index=False)
summary.to_csv(OUTPUT_DIR/'conversion_rates.csv')

if USE_S3:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            s3.upload_file(str(path),BUCKET,OUTPUT_PREFIX+path.name)
            print(f'Subido: s3://{BUCKET}/{OUTPUT_PREFIX}{path.name}')
else:
    print(f'Resultados locales en {OUTPUT_DIR.resolve()}')

## 10. Preguntas para entrevista y conclusión

Redacta respuestas breves a:

1. ¿Cuál es la diferencia conceptual entre bootstrap y simulación Monte Carlo bajo H₀?
2. ¿Cuándo preferirías un intervalo Newcombe frente a un bootstrap percentil?
3. ¿Qué significa potencia del 80% si la MDE es +1 pp?
4. ¿Por qué no podemos interpretar una ausencia de significancia como prueba de igualdad?
5. ¿Qué riesgos genera realizar 20 pruebas estadísticas y reportar solamente la de menor p-value?

### Plantilla de conclusión
> Estimamos una diferencia de **[X] pp** entre tratamiento y control. El intervalo bootstrap del 95% fue **[L, U] pp** y el p-value Monte Carlo bajo H₀ fue **[p]**. Estos resultados están sujetos a la independencia de observaciones, a la validez de las asignaciones y a las exclusiones de la muestra depurada. Para un experimento futuro con MDE de **[MDE] pp**, potencia **[power]** y α = **[alpha]**, se requieren aproximadamente **[n] usuarios por grupo**.

**Siguiente notebook:** 05 — Informe ejecutivo, decisión de despliegue y entrega final para portafolio.